In [1]:
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
import logging
from typing import List
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torch.nn as nn
import os
from utils.data.LandsatDataModule import LandsatDataModule

'''
-Wandb
-With/w/o pretrained weights
-By City, by geography
-Model selections
-Adding aug/reg
-Adding normalization
'''

class LSTNowcaster(pl.LightningModule):
    def __init__(self, in_channels=5, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = PixelwiseRegressionTask(
            model="unet",
            backbone="resnet50",
            weights=True,
            in_channels=in_channels,
            num_outputs=1,
            loss="mse",
            lr=learning_rate
        )
        self.criterion = nn.MSELoss()
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        loss = self.criterion(outputs[mask], targets[mask])

        self.log('train_loss', loss,
                 on_step=False,
                 on_epoch=True,
                 prog_bar=True,
                 sync_dist=True)  # Add this parameter
        return loss

    def validation_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        mse_loss = self.criterion(outputs[mask], targets[mask])
        
        mse_f = torch.mean((outputs[mask] - targets[mask])**2)
        rmse_f = torch.sqrt(mse_f)
        
        self.log(f"Val_RMSE: {rmse_f:.2f}°F", rmse_f, on_step=False, on_epoch=True, prog_bar=True)
        return {'val_loss': mse_loss, 'val_rmse': rmse_f}

    def test_step(self, batch, batch_idx):
        pass

    def on_test_epoch_end(self, outputs):
        pass
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

# Initialize data module
data_module = LandsatDataModule(
    data_dir="./Data",
    batch_size=1,
    num_workers=0,
    byCity=False,
    debug=True
)

# Initialize trainer with explicit steps
trainer = pl.Trainer(
    max_epochs=3,
    gradient_clip_val=0.5,
    log_every_n_steps=10,
    enable_progress_bar=True,
    enable_model_summary=False,
    deterministic=True,
    num_sanity_val_steps=2,
    reload_dataloaders_every_n_epochs=1
)

/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /work/ubh496/.conda/envs/ml3/lib/python3.10/site-pac ...
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [2]:
model = LSTNowcaster(in_channels=5, learning_rate=1e-4)

/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/torch/hub.py:846: UserWarning: TORCH_MODEL_ZOO is deprecated, please use env TORCH_HOME instead
  warnings.warn(


In [ ]:
trainer.fit(model=model, datamodule=data_module)

Preparing scene by scene...: 100%|██████████| 650/650 [00:00<00:00, 2082.83it/s]
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to improve performance.
/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

In [ ]:
trainer.test(model=model, datamodule=data_module)